In [2]:
#
import pandas as pd
pd.set_option('display.max_columns', None)

# 
import numpy as np

# 
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

# 
import warnings
warnings.filterwarnings('ignore')

In [3]:
df = pd.read_csv('files/hr.csv')

In [3]:
df.head(5)

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,EnvironmentSatisfaction,Gender,HourlyRate,JobInvolvement,JobLevel,JobRole,JobSatisfaction,MaritalStatus,MonthlyIncome,MonthlyRate,NumCompaniesWorked,Over18,OverTime,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41.0,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,2,Female,94,3,2,sALES eXECUTIVE,4.0,Single,5993.0,19479,8,Y,Yes,11,3,1,80.0,0,8,0.0,1,6,4,0,5.0
1,49.0,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,3,Male,61,2,2,rESEARCH sCIENTIST,2.0,Married,5130.0,24907,1,Y,No,23,4,4,NaN,1,10,3.0,3,10,7,1,7.0
2,37.0,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,4,Male,92,2,1,lABORATORY tECHNICIAN,3.0,Single,2090.0,2396,6,Y,Yes,15,3,2,NaN,0,7,3.0,3,0,0,0,0.0
3,33.0,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,4,Female,56,3,1,rESEARCH sCIENTIST,3.0,Married,2909.0,23159,1,Y,Yes,11,3,3,80.0,0,8,3.0,3,8,7,3,0.0
4,27.0,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,1,Male,40,3,1,lABORATORY tECHNICIAN,2.0,Married,3468.0,16632,9,Y,No,12,3,4,80.0,1,6,3.0,3,2,2,2,2.0


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1474 entries, 0 to 1473
Data columns (total 35 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Age                       1401 non-null   float64
 1   Attrition                 1474 non-null   str    
 2   BusinessTravel            1357 non-null   str    
 3   DailyRate                 1474 non-null   int64  
 4   Department                1445 non-null   str    
 5   DistanceFromHome          1474 non-null   int64  
 6   Education                 1474 non-null   int64  
 7   EducationField            1416 non-null   str    
 8   EmployeeCount             1474 non-null   int64  
 9   EmployeeNumber            1474 non-null   int64  
 10  EnvironmentSatisfaction   1474 non-null   int64  
 11  Gender                    1474 non-null   str    
 12  HourlyRate                1474 non-null   int64  
 13  JobInvolvement            1474 non-null   int64  
 14  JobLevel           

In [5]:
df.shape

(1474, 35)

In [6]:
# Se busca las columnas que tienen valores nulos
nulos = df.isnull().sum()
nulos = nulos[nulos>0]
nulos

Age                       73
BusinessTravel           117
Department                29
EducationField            58
JobSatisfaction           29
MaritalStatus            132
MonthlyIncome             14
OverTime                  44
StandardHours            164
TrainingTimesLastYear     88
YearsWithCurrManager     148
dtype: int64

In [7]:
# Porcentaje de nulos de dichas columnas
nulos_pct = round(df.isnull().sum()/df.shape[0]*100, 2)
nulos_pct = nulos_pct[nulos_pct>0]
nulos_pct

Age                       4.95
BusinessTravel            7.94
Department                1.97
EducationField            3.93
JobSatisfaction           1.97
MaritalStatus             8.96
MonthlyIncome             0.95
OverTime                  2.99
StandardHours            11.13
TrainingTimesLastYear     5.97
YearsWithCurrManager     10.04
dtype: float64

In [8]:
# Se comprueba si existen columnas duplicadas y se verifica que los numeros de empleados están repetidos
print(df.duplicated().sum())
print(df['EmployeeNumber'].duplicated().sum())

4
4


In [9]:
# Mostramos las filas con EmployeeNumber duplicado
# y las ordenamos por EmployeeNumber para ver juntas las repeticiones.
df[df['EmployeeNumber'].duplicated(keep=False)].sort_values('EmployeeNumber')

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,EnvironmentSatisfaction,Gender,HourlyRate,JobInvolvement,JobLevel,JobRole,JobSatisfaction,MaritalStatus,MonthlyIncome,MonthlyRate,NumCompaniesWorked,Over18,OverTime,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
67,45.0,No,Travel_Rarely,1339,Research & Development,7,3,Life Sciences,1,86,2,Male,59,3,3,rESEARCH sCIENTIST,1.0,Divorced,9724.0,18787,2,Y,No,17,3,3,NaN,1,25,2.0,3,1,0,0,0.0
1473,45.0,No,Travel_Rarely,1339,Research & Development,7,3,Life Sciences,1,86,2,Male,59,3,3,rESEARCH sCIENTIST,1.0,Divorced,9724.0,18787,2,Y,No,17,3,3,NaN,1,25,2.0,3,1,0,0,0.0
1471,53.0,No,Travel_Rarely,1084,Research & Development,13,2,Medical,1,250,4,Female,57,4,2,mANUFACTURING dIRECTOR,1.0,Divorced,4450.0,26250,1,Y,No,11,3,3,NaN,2,5,3.0,3,4,2,1,3.0
184,53.0,No,Travel_Rarely,1084,Research & Development,13,2,Medical,1,250,4,Female,57,4,2,mANUFACTURING dIRECTOR,1.0,Divorced,4450.0,26250,1,Y,No,11,3,3,NaN,2,5,3.0,3,4,2,1,3.0
1470,28.0,No,Travel_Rarely,866,Sales,5,3,Medical,1,1469,4,Male,84,3,2,sALES eXECUTIVE,1.0,Single,8463.0,23490,0,Y,No,18,3,4,NaN,0,6,4.0,3,5,4,1,NaN
1041,28.0,No,Travel_Rarely,866,Sales,5,3,Medical,1,1469,4,Male,84,3,2,sALES eXECUTIVE,1.0,Single,8463.0,23490,0,Y,No,18,3,4,NaN,0,6,4.0,3,5,4,1,NaN
1222,24.0,Yes,Travel_Rarely,240,Human Resources,22,1,Human Resources,1,1714,4,Male,58,1,1,hUMAN rESOURCES,3.0,Married,1555.0,11585,1,Y,No,11,3,3,80.0,1,1,2.0,3,1,0,0,0.0
1472,24.0,Yes,Travel_Rarely,240,Human Resources,22,1,Human Resources,1,1714,4,Male,58,1,1,hUMAN rESOURCES,3.0,Married,1555.0,11585,1,Y,No,11,3,3,80.0,1,1,2.0,3,1,0,0,0.0


In [10]:
# Se verifica si hay negativos en las columnas
(df.select_dtypes(include='number') < 0).any().any()

np.False_

Se verifican varias cosas hasta este punto:
* Los títulos de columnas siguen la misma estética. No es necesario cambiarlos.
* JobRol tiene los textos con las mayúsculas invertidas.
* Hay 4 filas duplicadas. Confirmado en que el número de empleado se repite que debería ser un valor único.
* No hay números negativos en ninguna columna.

In [11]:
# Se comprueba que los valores de Attrition sean solo "Yes" o "No". No tiene nulos. Convertir a Booleano
df['Attrition'].unique()

<StringArray>
['Yes', 'No']
Length: 2, dtype: str

In [12]:
# Education es un valor numérico que mide la escala de estudios, se comprueba los valores comprendidos. No tiene nulos
df['Education'].unique()

array([2, 1, 4, 3, 5])

In [13]:
# Solo debe devolver valor "1". Se puede descartar de la lista. No aporta valor
df['EmployeeCount'].unique()

array([1])

In [14]:
# Encuesta de satisfacción. Escala de valores entre 1 y 4. No tiene nulos
df['EnvironmentSatisfaction'].unique()

array([2, 3, 4, 1])

In [15]:
# Dos valores. No tiene nulos
df['Gender'].unique()

<StringArray>
['Female', 'Male']
Length: 2, dtype: str

In [16]:
# Encuesta de satisfacción. Escala de valores entre 1 y 4. No tiene nulos
df['JobInvolvement'].unique()

array([3, 2, 4, 1])

In [17]:
# Jerarquía de la empresa. Valores entre 1 y 5. No tiene nulos
df['JobLevel'].unique()

array([2, 1, 3, 4, 5])

In [18]:
# Encuesta de satisfacción. Escala de valores entre 1 y 4. Tiene nulos
df['JobSatisfaction'].unique()

array([ 4.,  2.,  3.,  1., nan])

In [19]:
# Cuatro valores y nulos
df['MaritalStatus'].unique()

<StringArray>
['Single', 'Married', 'Divorced', nan, 'Marreid']
Length: 5, dtype: str

In [20]:
# Valores desde el 0 hasta el 9
df['NumCompaniesWorked'].unique()

array([8, 1, 6, 9, 0, 4, 5, 2, 7, 3])

In [21]:
# Se puede convertir a booleano o descartar de la lista. No aporta valor, todos los empleados son mayores de 18 años
df['Over18'].unique()

<StringArray>
['Y']
Length: 1, dtype: str

In [22]:
# Una vez cubiertos los nulos, se puede convertir a booleano
df['OverTime'].unique()

<StringArray>
['Yes', 'No', nan]
Length: 3, dtype: str

In [23]:
# Valor de empresa respecto al empleado. Escala de valores entre 1 y 4, posible descarte. No conecta con la satisfacción del empleado
df['PerformanceRating'].unique()

array([3, 4])

In [24]:
# Encuesta de satisfacción. Escala de valores entre 1 y 4. Sin nulos
df['RelationshipSatisfaction'].unique()

array([1, 4, 2, 3])

In [25]:
# Jornada Laboral. Todos los empleados tienen el mismo número de horas o si no el valor es nulo. Posible descarte de la lista. No aporta valor
df['StandardHours'].unique()

array([80., nan])

In [26]:
# Valores entre 0 y 3. Valor 0 indica sin acciones. Valor 1 nivel alto de acciones. Valor 2 nivel medio de acciones. Valor 3 nivel bajo de acciones
df['StockOptionLevel'].unique()

array([0, 1, 3, 2])

In [27]:
# Valor de formaciones que se ha recibido en el último año. Escala de valores entre 0 y 6. Con nulos. Convertir a INT al retirar nulos
df['TrainingTimesLastYear'].unique()

array([ 0.,  3.,  2.,  5.,  1.,  4., nan,  6.])

In [28]:
# Encuesta de satisfacción. Escala de valores entre 1 y 4. Sin nulos
df['WorkLifeBalance'].unique()

array([1, 3, 2, 4])

Se identifican las siguientes cuestiones:

* Las columnas `Attrition`, `Over18` y `OverTime` son valores booleanos en STR. No siguen el mismo formato.
* Las columnas `EmployeeCount`, `Over18`, `PerformanceRating` y `StandardHours` no aportan valor claro al estudio. Es necesario determinar si se descartan valores o se quieren conservar.
* Hay columnas en float que deben pasar a INT: `Age`, `JobSatisfaction`, `StandardHours`, `TrainingTimesLastYear` y `YearsWithCurrManager`.
* Se comprueba que las columnas con encuestas de satisfacción siempre tienen valores del 1 al 4.

## Limpieza y transformación de datos


In [29]:
#Convertimos la columna Age a número entero
## Usamos Int64 en vez de int porque Age tiene valores nulos.
# El tipo Int64 permite trabajar con enteros y mantener los nulos como <NA>.
df['Age'] = df['Age'].astype('Int64')
df['Age'].dtype

Int64Dtype()

In [30]:
#Convertimos la columna JobSatisfaction a número entero
## Usamos Int64 en vez de int porque JobSatisfaction tiene valores nulos.
# El tipo Int64 permite trabajar con enteros y mantener los nulos como <NA>.
df['JobSatisfaction'] = df['JobSatisfaction'].astype('Int64')
df['JobSatisfaction'].dtype

Int64Dtype()

In [31]:
#Convertimos la columna StandardHours a número entero, aunque se contempla el descarte
## Usamos Int64 en vez de int porque StandardHours tiene valores nulos.
# El tipo Int64 permite trabajar con enteros y mantener los nulos como <NA>.
df['StandardHours'] = df['StandardHours'].astype('Int64')
df['StandardHours'].dtype

Int64Dtype()

In [32]:
#Convertimos la columna TrainingTimesLastYear a número entero
## Usamos Int64 en vez de int porque TrainingTimesLastYear tiene valores nulos.
# El tipo Int64 permite trabajar con enteros y mantener los nulos como <NA>.
df['TrainingTimesLastYear'] = df['TrainingTimesLastYear'].astype('Int64')
df['TrainingTimesLastYear'].dtype

Int64Dtype()

In [33]:
#Convertimos la columna YearsWithCurrManager a número entero
## Usamos Int64 en vez de int porque YearsWithCurrManager tiene valores nulos.
# El tipo Int64 permite trabajar con enteros y mantener los nulos como <NA>.
df['YearsWithCurrManager'] = df['YearsWithCurrManager'].astype('Int64')
df['YearsWithCurrManager'].dtype

Int64Dtype()

In [34]:
# Convertimos JobRole a minúsculas y eliminamos espacios sobrantes
# al principio y al final de cada valor.
df['JobRole'] = df['JobRole'].str.strip().str.title()
df.head(5)

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,EnvironmentSatisfaction,Gender,HourlyRate,JobInvolvement,JobLevel,JobRole,JobSatisfaction,MaritalStatus,MonthlyIncome,MonthlyRate,NumCompaniesWorked,Over18,OverTime,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,2,Female,94,3,2,Sales Executive,4,Single,5993.0,19479,8,Y,Yes,11,3,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,3,Male,61,2,2,Research Scientist,2,Married,5130.0,24907,1,Y,No,23,4,4,<NA>,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,4,Male,92,2,1,Laboratory Technician,3,Single,2090.0,2396,6,Y,Yes,15,3,2,<NA>,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,4,Female,56,3,1,Research Scientist,3,Married,2909.0,23159,1,Y,Yes,11,3,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,1,Male,40,3,1,Laboratory Technician,2,Married,3468.0,16632,9,Y,No,12,3,4,80,1,6,3,3,2,2,2,2


In [35]:
# 1. Comprobamos los valores únicos actuales de la columna Attrition
# Deberían aparecer únicamente los valores 'Yes' y 'No'.
df['Attrition'].unique()

<StringArray>
['Yes', 'No']
Length: 2, dtype: str

In [36]:
# 2. Convertimos la columna Attrition a valores booleanos
# 'Yes' se transforma en True
# 'No' se transforma en False
df['Attrition'] = df['Attrition'].map({
    'Yes': True,
    'No': False
}).astype('boolean')
df['Attrition'].unique()

<BooleanArray>
[True, False]
Length: 2, dtype: boolean

In [37]:
df['Attrition'].dtype

BooleanDtype

In [38]:
# Convertimos la columna Over18 a valores booleanos
# 'Y' se transforma en True. No contiene nulos.
df['Over18'] = df['Over18'].map({'Y': True, 'N': False}).astype('boolean')

In [39]:
df['Over18'].dtype

BooleanDtype

In [40]:
df['OverTime'] = df['OverTime'].map({'Yes': True, 'No': False}).astype('boolean')

In [41]:
df['OverTime'].dtype

BooleanDtype

In [42]:
# Eliminamos las filas completamente duplicadas
df = df.drop_duplicates(keep='first')
print(df.duplicated().sum())


0


In [43]:
# EmployeeNumber identifica de forma única a cada empleado/a,
# por lo que lo usamos como índice del dataframe.
df = df.set_index('EmployeeNumber')

In [44]:
df.head(10)

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EnvironmentSatisfaction,Gender,HourlyRate,JobInvolvement,JobLevel,JobRole,JobSatisfaction,MaritalStatus,MonthlyIncome,MonthlyRate,NumCompaniesWorked,Over18,OverTime,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
EmployeeNumber,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1,41,True,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,2,Female,94,3,2,Sales Executive,4,Single,5993.0,19479,8,True,True,11,3,1,80,0,8,0,1,6,4,0,5
2,49,False,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,3,Male,61,2,2,Research Scientist,2,Married,5130.0,24907,1,True,False,23,4,4,<NA>,1,10,3,3,10,7,1,7
4,37,True,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,Male,92,2,1,Laboratory Technician,3,Single,2090.0,2396,6,True,True,15,3,2,<NA>,0,7,3,3,0,0,0,0
5,33,False,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,4,Female,56,3,1,Research Scientist,3,Married,2909.0,23159,1,True,True,11,3,3,80,0,8,3,3,8,7,3,0
7,27,False,Travel_Rarely,591,Research & Development,2,1,Medical,1,1,Male,40,3,1,Laboratory Technician,2,Married,3468.0,16632,9,True,False,12,3,4,80,1,6,3,3,2,2,2,2
8,32,False,Travel_Frequently,1005,Research & Development,2,2,Life Sciences,1,4,Male,79,3,1,Laboratory Technician,4,Single,3068.0,11864,0,True,False,13,3,3,80,0,8,2,2,7,7,3,6
10,59,False,Travel_Rarely,1324,Research & Development,3,3,Medical,1,3,Female,81,4,1,Laboratory Technician,1,Married,2670.0,9964,4,True,<NA>,20,4,1,<NA>,3,12,3,2,1,0,0,<NA>
11,30,False,Travel_Rarely,1358,NaN,24,1,Life Sciences,1,4,Male,67,3,1,Laboratory Technician,3,Divorced,2693.0,13335,1,True,False,22,4,2,80,1,1,2,3,1,0,0,0
12,38,False,Travel_Frequently,216,Research & Development,23,3,Life Sciences,1,4,Male,44,2,3,Manufacturing Director,3,Single,9526.0,8787,0,True,False,21,4,2,80,0,10,2,3,9,7,1,8


In [4]:
df['MaritalStatus'].value_counts()

MaritalStatus
Married     604
Single      437
Divorced    298
Marreid       3
Name: count, dtype: int64

In [5]:
df['MaritalStatus'] = df['MaritalStatus'].replace('Marreid', 'Married')

In [6]:
df['MaritalStatus'].value_counts()

MaritalStatus
Married     607
Single      437
Divorced    298
Name: count, dtype: int64

## Avances hasta este punto:
- Transformación de `Age`, `JobSatisfaction`, `StandardHours`, `TrainingTimesLastYear` y `YearsWithCurrManager` a números enteros con formato int64 para que siga reconociendo los nulos y luego hacer la gestión de nulos
- Normalización de los valores en la columna JobRole
- Transformación a booleano de columna `Attrition` (que no tenía nulos), `Over18` y `OverTime`
- Tras comprobar que las filas repetidas eran exactamente iguales, se han eliminado las duplicadas quedándonos con la primera, y después EmployeeNumber se ha fijado como index de la tabla

- En MaritalStatus aparecía en tres casos Marreid: se considera error al insertar los datos. Se reemplaza por Married

#### Columnas interesantes para el análisis

PercentSalaryHike es el porcentaje de aumento salarial, seguramente desde la última revisión. Puede ser una variable interesante para el análisis

In [45]:
df.info()

<class 'pandas.DataFrame'>
Index: 1470 entries, 1 to 2068
Data columns (total 34 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Age                       1397 non-null   Int64  
 1   Attrition                 1470 non-null   boolean
 2   BusinessTravel            1353 non-null   str    
 3   DailyRate                 1470 non-null   int64  
 4   Department                1441 non-null   str    
 5   DistanceFromHome          1470 non-null   int64  
 6   Education                 1470 non-null   int64  
 7   EducationField            1412 non-null   str    
 8   EmployeeCount             1470 non-null   int64  
 9   EnvironmentSatisfaction   1470 non-null   int64  
 10  Gender                    1470 non-null   str    
 11  HourlyRate                1470 non-null   int64  
 12  JobInvolvement            1470 non-null   int64  
 13  JobLevel                  1470 non-null   int64  
 14  JobRole                 

In [ ]:
## FUNCION PARA SUSTITUIR NULOS EN COLUMNAS CATEGORICAs - PY

def sustitucion_nulos_categoricas(df):
    df = df.copy()
    columnas_categoricas = df.select_dtypes(include='object').columns
    columnas_con_nulos = [col for col in columnas_categoricas if df[col].isnull().sum() > 0]
    for columna in columnas_con_nulos:
        porcentaje_nulos = df[columna].isnull().sum()/df.shape[0]*100
        print(f"{columna}. Nulos antes = {df[columna].isnull().sum()} ({porcentaje_nulos:.1f}%)")

        lista_valores = df[columna].value_counts()/df.shape[0]*100
        dominante = lista_valores.iloc[0]
        if porcentaje_nulos > 25:
            df[columna] = df[columna].fillna(lista_valores.index[0])
            print(f"{columna}. Nulos después = {df[columna].isnull().sum()}")

        else:
            print(f"{columna}. Dominante = {dominante:.1f}%")
            if dominante <= 75:
                df[columna] = df[columna].fillna('unknown')
                print(f"{columna}. Nulos después = {df[columna].isnull().sum()}")
            else:
                df[columna] = df[columna].fillna(lista_valores.index[0])
                print(f"{columna}. Nulos después = {df[columna].isnull().sum()}")

    return df

In [47]:
df_cat_limpio = sustitucion_nulos_categoricas(df)
df_cat_limpio

BusinessTravel. Nulos antes = 117 (8.0%)
BusinessTravel. Dominante = 64.7%
BusinessTravel. Nulos después = 0
Department. Nulos antes = 29 (2.0%)
Department. Dominante = 63.9%
Department. Nulos después = 0
EducationField. Nulos antes = 58 (3.9%)
EducationField. Dominante = 39.5%
EducationField. Nulos después = 0
MaritalStatus. Nulos antes = 132 (9.0%)
MaritalStatus. Dominante = 41.0%
MaritalStatus. Nulos después = 0


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EnvironmentSatisfaction,Gender,HourlyRate,JobInvolvement,JobLevel,JobRole,JobSatisfaction,MaritalStatus,MonthlyIncome,MonthlyRate,NumCompaniesWorked,Over18,OverTime,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
EmployeeNumber,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1,41,True,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,2,Female,94,3,2,Sales Executive,4,Single,5993.0,19479,8,True,True,11,3,1,80,0,8,0,1,6,4,0,5
2,49,False,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,3,Male,61,2,2,Research Scientist,2,Married,5130.0,24907,1,True,False,23,4,4,<NA>,1,10,3,3,10,7,1,7
4,37,True,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,Male,92,2,1,Laboratory Technician,3,Single,2090.0,2396,6,True,True,15,3,2,<NA>,0,7,3,3,0,0,0,0
5,33,False,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,4,Female,56,3,1,Research Scientist,3,Married,2909.0,23159,1,True,True,11,3,3,80,0,8,3,3,8,7,3,0
7,27,False,Travel_Rarely,591,Research & Development,2,1,Medical,1,1,Male,40,3,1,Laboratory Technician,2,Married,3468.0,16632,9,True,False,12,3,4,80,1,6,3,3,2,2,2,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2061,36,False,Travel_Frequently,884,Research & Development,23,2,Medical,1,3,Male,41,4,2,Laboratory Technician,4,Married,2571.0,12290,4,True,False,17,3,3,80,1,17,3,3,5,2,0,3
2062,39,False,Travel_Rarely,613,Research & Development,6,1,Medical,1,4,Male,42,2,3,Healthcare Representative,1,Married,9991.0,21457,4,True,False,15,3,1,80,1,9,5,3,7,7,1,7
2064,27,False,Travel_Rarely,155,Research & Development,4,3,Life Sciences,1,2,Male,87,4,2,Manufacturing Director,2,Married,6142.0,5174,1,True,True,20,4,2,80,1,6,0,3,6,2,0,3


In [52]:
## Funcion para booleanas
def sustitucion_nulos_booleanas(df):
    df = df.copy()
    columnas_bool = df.select_dtypes(include='boolean').columns
    columnas_con_nulos = [col for col in columnas_bool if df[col].isnull().sum() > 0]

    for columna in columnas_con_nulos:
        moda = df[columna].mode()[0]
        print(f"{columna}: nulos antes = {df[columna].isnull().sum()}. Se sustituye con la moda: ({moda})")
        df[columna] = df[columna].fillna(moda)

    return df

In [53]:
df_bool_limpio = sustitucion_nulos_booleanas(df_cat_limpio)
df_bool_limpio

OverTime: nulos antes = 44. Se sustituye con la moda: (False)


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EnvironmentSatisfaction,Gender,HourlyRate,JobInvolvement,JobLevel,JobRole,JobSatisfaction,MaritalStatus,MonthlyIncome,MonthlyRate,NumCompaniesWorked,Over18,OverTime,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
EmployeeNumber,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1,41,True,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,2,Female,94,3,2,Sales Executive,4,Single,5993.0,19479,8,True,True,11,3,1,80,0,8,0,1,6,4,0,5
2,49,False,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,3,Male,61,2,2,Research Scientist,2,Married,5130.0,24907,1,True,False,23,4,4,<NA>,1,10,3,3,10,7,1,7
4,37,True,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,Male,92,2,1,Laboratory Technician,3,Single,2090.0,2396,6,True,True,15,3,2,<NA>,0,7,3,3,0,0,0,0
5,33,False,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,4,Female,56,3,1,Research Scientist,3,Married,2909.0,23159,1,True,True,11,3,3,80,0,8,3,3,8,7,3,0
7,27,False,Travel_Rarely,591,Research & Development,2,1,Medical,1,1,Male,40,3,1,Laboratory Technician,2,Married,3468.0,16632,9,True,False,12,3,4,80,1,6,3,3,2,2,2,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2061,36,False,Travel_Frequently,884,Research & Development,23,2,Medical,1,3,Male,41,4,2,Laboratory Technician,4,Married,2571.0,12290,4,True,False,17,3,3,80,1,17,3,3,5,2,0,3
2062,39,False,Travel_Rarely,613,Research & Development,6,1,Medical,1,4,Male,42,2,3,Healthcare Representative,1,Married,9991.0,21457,4,True,False,15,3,1,80,1,9,5,3,7,7,1,7
2064,27,False,Travel_Rarely,155,Research & Development,4,3,Life Sciences,1,2,Male,87,4,2,Manufacturing Director,2,Married,6142.0,5174,1,True,True,20,4,2,80,1,6,0,3,6,2,0,3


In [54]:
## FUNCION PARA SUSTITUIR NULOS EN COLUMNAS NUMERICAS - PY

def sustitucion_nulos_numericas(df):
    df = df.copy()
    columnas_numericas = df.select_dtypes(include=['number']).columns
    columnas_con_nulos = [col for col in columnas_numericas if df[col].isnull().sum() > 0]
    imputador = IterativeImputer(max_iter= 10, random_state= 42)
    valores_imputados = imputador.fit_transform(df[columnas_numericas])
    df_temp = pd.DataFrame(valores_imputados, columns=columnas_numericas, index=df.index)
    
    for columna in columnas_con_nulos:
        porcentaje_nulos = df[columna].isnull().sum()/df.shape[0]*100
        print(f"{columna}. Nulos antes = {df[columna].isnull().sum()} ({porcentaje_nulos:.1f}%)")


        if porcentaje_nulos > 25:
            df[columna] = df_temp[columna]
            print(f"{columna}. Nulos después = {df[columna].isnull().sum()}")

        else:
            sesgo = df[columna].skew()
            mediana = df[columna].median()
            media = df[columna].mean() 
            print(f"{columna}. Sesgo = {sesgo:.2f}")
            if abs(sesgo) > 0.5:
                df[columna] = df[columna].fillna(round(mediana))
                print(f"{columna}. Nulos después = {df[columna].isnull().sum()}")
            else:
                df[columna] = df[columna].fillna(round(media))
                print(f"{columna}. Nulos después = {df[columna].isnull().sum()}")

    return df

In [55]:
df_limpio = sustitucion_nulos_numericas(df_bool_limpio)
df_limpio

Age. Nulos antes = 73 (5.0%)
Age. Sesgo = 0.41
Age. Nulos después = 0
JobSatisfaction. Nulos antes = 29 (2.0%)
JobSatisfaction. Sesgo = -0.33
JobSatisfaction. Nulos después = 0
MonthlyIncome. Nulos antes = 14 (1.0%)
MonthlyIncome. Sesgo = 1.37
MonthlyIncome. Nulos después = 0
StandardHours. Nulos antes = 161 (11.0%)
StandardHours. Sesgo = 0.00
StandardHours. Nulos después = 0
TrainingTimesLastYear. Nulos antes = 88 (6.0%)
TrainingTimesLastYear. Sesgo = 0.55
TrainingTimesLastYear. Nulos después = 0
YearsWithCurrManager. Nulos antes = 147 (10.0%)
YearsWithCurrManager. Sesgo = 0.85
YearsWithCurrManager. Nulos después = 0


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EnvironmentSatisfaction,Gender,HourlyRate,JobInvolvement,JobLevel,JobRole,JobSatisfaction,MaritalStatus,MonthlyIncome,MonthlyRate,NumCompaniesWorked,Over18,OverTime,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
EmployeeNumber,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1,41,True,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,2,Female,94,3,2,Sales Executive,4,Single,5993.0,19479,8,True,True,11,3,1,80,0,8,0,1,6,4,0,5
2,49,False,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,3,Male,61,2,2,Research Scientist,2,Married,5130.0,24907,1,True,False,23,4,4,80,1,10,3,3,10,7,1,7
4,37,True,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,Male,92,2,1,Laboratory Technician,3,Single,2090.0,2396,6,True,True,15,3,2,80,0,7,3,3,0,0,0,0
5,33,False,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,4,Female,56,3,1,Research Scientist,3,Married,2909.0,23159,1,True,True,11,3,3,80,0,8,3,3,8,7,3,0
7,27,False,Travel_Rarely,591,Research & Development,2,1,Medical,1,1,Male,40,3,1,Laboratory Technician,2,Married,3468.0,16632,9,True,False,12,3,4,80,1,6,3,3,2,2,2,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2061,36,False,Travel_Frequently,884,Research & Development,23,2,Medical,1,3,Male,41,4,2,Laboratory Technician,4,Married,2571.0,12290,4,True,False,17,3,3,80,1,17,3,3,5,2,0,3
2062,39,False,Travel_Rarely,613,Research & Development,6,1,Medical,1,4,Male,42,2,3,Healthcare Representative,1,Married,9991.0,21457,4,True,False,15,3,1,80,1,9,5,3,7,7,1,7
2064,27,False,Travel_Rarely,155,Research & Development,4,3,Life Sciences,1,2,Male,87,4,2,Manufacturing Director,2,Married,6142.0,5174,1,True,True,20,4,2,80,1,6,0,3,6,2,0,3


In [58]:
df_limpio.isnull().any().any()

np.False_

In [57]:
df_limpio.info()

<class 'pandas.DataFrame'>
Index: 1470 entries, 1 to 2068
Data columns (total 34 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Age                       1470 non-null   Int64  
 1   Attrition                 1470 non-null   boolean
 2   BusinessTravel            1470 non-null   str    
 3   DailyRate                 1470 non-null   int64  
 4   Department                1470 non-null   str    
 5   DistanceFromHome          1470 non-null   int64  
 6   Education                 1470 non-null   int64  
 7   EducationField            1470 non-null   str    
 8   EmployeeCount             1470 non-null   int64  
 9   EnvironmentSatisfaction   1470 non-null   int64  
 10  Gender                    1470 non-null   str    
 11  HourlyRate                1470 non-null   int64  
 12  JobInvolvement            1470 non-null   int64  
 13  JobLevel                  1470 non-null   int64  
 14  JobRole                 

Se añaden cambios en la parte anterior y se realizan tres funciones para imputar los nulos